### Install New Libraries

In [ ]:
#!pip install ddgs trafilatura -q
#!pip install openai-agents

### Setup

In [ ]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura

from agents import Agent, Runner, function_tool

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [ ]:
@function_tool
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Returns 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got Results for {query}\n")
    return json.dumps(results, indent=2)

In [ ]:
@function_tool
def fetch_url(url: str):
    """Fetch the URL content using Trafilatura and extract the text. Returns the extracted text if successful, otherwise a failure message."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"  \u2705 Got text: {len(text)} characters\n")
            return text
    print(f"  \u274C Failed to fetch or extracte text from {url}\n")
    return f"Could not fetch or extract text from {url}. Try a different source."

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce

#### Research Agent

In [ ]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief.
If you have fewer than 6 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

#### Orchestrator Agent

In [ ]:
ORCHESTRATOR_AGENT_PROMPT = """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
"""

orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning for choosing this model is that it is lightweight and fast, suitable for orchestrating tasks without heavy computational overhead
    tools=[research_agent.as_tool(tool_name="research_agent", tool_description="A research specialist agent that gathers information and produces comprehensive research briefs.")]
)

### Let's Run It!

In [ ]:
result = await Runner.run(
    orchestrator_agent,
    input = "Research the following topic and produce a comprehensive research brief: how will AI be used in Healthcare in 2040?",
    max_turns=30
)

In [ ]:
print(f"Agent: {result.last_agent.name}")
print(f"----------------")
display(Markdown(result.final_output))